# NorthwoodFresh Data: Summary Statistics

Module 1 Project

Author: John Ko
Date: 10/13/26

In [78]:
import pandas as pd
import numpy as np
import scipy 
from scipy import stats


In [ ]:
# Loading the dataset
northwoodfresh = pd.read_csv("northwoodfresh_sales.csv")
northwoodfresh.head(10)

,region,month,revenue
0,Northeast,1,532914.57
1,Northeast,2,516405.13
2,Northeast,3,536839.90
3,Northeast,4,559598.78
4,Northeast,5,513912.01
5,Northeast,6,513912.44
6,Northeast,7,561059.53
7,Northeast,8,539953.30
8,Northeast,9,507793.67
9,Northeast,10,534106.56


In [ ]:
# Displaying the shape and column types of the dataset
print(f"\nShape: {northwoodfresh.shape}")
print(f"\nColumn types:\n{northwoodfresh.dtypes}")


Shape: (48, 3)

Column types:
region         str
month        int64
revenue    float64
dtype: object


## Connecting the northwoodfresh_sales

Shapes reveal that there are 48 rows or entries with three columns
Region - a string of text of where the location is
Month - Integer of which month 
Revenue - A float or continous data of how much money money was made that month

In [ ]:
# Calculating IQR for each region
grouped_iqr = northwoodfresh.groupby('region')['revenue'].agg(iqr)
print(grouped_iqr)

# Calculating summary statistics for each region
northwoodfresh.groupby("region")["revenue"].agg(
    mean="mean",
    median="median",
    std="std", 
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(2)

region
Midwest       38927.0625
Northeast     25196.4575
Southeast    120574.0675
West          26338.3175
Name: revenue, dtype: float64


,mean,median,std,iqr
region,,,,
Midwest,494923.59,491253.53,25492.22,38927.06
Northeast,527694.84,524659.85,19350.27,25196.46
Southeast,417943.34,411262.34,60019.93,120574.07
West,584296.61,577667.87,26461.74,26338.32


## Calculating Summary Statistic by Region

Calculate the meanm median, standard deviation and IQR by region using agg function

The Southeast has the biggest IQR and standard deviation due to the first 6 months having an average of 480,000 while the latter 6 months having an average of 370,000. Southeast mean is lying due to the data being a bimodality

All the remaining region having mean and median that are pretty similiar and is reliable



In [ ]:
# Creating a histogram for the total revenue earned by Southeast region
import plotly.express as px

southeast = northwoodfresh[northwoodfresh["region"] == "Southeast"]

fig = px.histogram(
    southeast,
    x="revenue",
    nbins=10,
    title="Distribution of Revenue (USD)",
)
fig.show()

In [ ]:
#  Creating a box plot for the total revenue earned by region
fig = px.box(
    northwoodfresh,
    x="region",
    y="revenue",
    title="Total Revenue Earned by Region (Box Plot)",
    points="all",
)
fig.show()

## Distribution Visualizations

The first one is a histogram of just the Southeast region. The data shows that the graph is a biomodality shaped. This is due to the data being centered around two different locations

The second one is box plot graph of each regions. The one with the biggest box (largest IQR) is Southeast with it being bimodality. The other regions are relatively small in comparison. With data being mostly symmetric with West having an extreme outlier to the right. 

In [98]:
#  Getting the sample size, first half, and second half of the Southeast region revenue data
se_first = southeast["revenue"][0:6]
se_second = southeast["revenue"][6: ]

# Calculating the mean of the first half and second half of the Southeast region revenue data
se_first_mean1, se_second_mean2 = se_first.mean(), se_second.mean()
se_mean_diff = se_first_mean1 - se_second_mean2

se_percentage_drop = se_mean_diff / se_first_mean1 * 100

# Calculating the standard deviation of the first half and second half of the Southeast region revenue data
se_first_std, se_second_std = se_first.std(), se_second.std()

# Calculating the standard deviation of the difference between the first half and second half of the Southeast region revenue data
se_std_diff = np.sqrt((se_first_std**2 / len(se_first)) + (se_second_std**2 / len(se_second)))

# Critical value for 95% confidence
z = 1.96

# Margin of error
margin_of_error = z * se_std_diff

# Confidence interval
ci_lower = se_mean_diff - margin_of_error
ci_upper = se_mean_diff + margin_of_error

ci_scipy = stats.norm.interval(
    confidence=0.95,
    loc=se_mean_diff,
    scale=se_std_diff,
)

# Calculating the t-based confidence interval
t_stat, p_value = stats.ttest_ind(se_first, se_second)



print(f'Southeast First Half: n={len(se_first)}, Mean=${se_first_mean1:.2f}, std=${se_first_std:.2f}')
print(f'Southeast Second Half: n={len(se_second)}, Mean=${se_second_mean2:.2f}, std=${se_second_std:.2f}')
print(f'Southeast Mean Difference: {se_mean_diff:.2f}')
print(f'Southeast Standard Deviation Difference: {se_std_diff:.2f}')

print(f"Manual 95% CI: (${ci_lower:.2f}, ${ci_upper:.2f})")
print(f"scipy 95% CI:  (${ci_scipy[0]:.2f}, ${ci_scipy[1]:.2f})")

print(f"t-based 95% CI: (${ci_t_lower:.2f}, ${ci_t_upper:.2f})")
print(f"t critical value at df={n-1}: {t_crit:.4f}")

print(f'Effect Size: ${se_mean_diff:.2f}')
print(f"Effect Size Percentage: {se_percentage_drop:.2f}%")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.7f}")

Southeast First Half: n=6, Mean=$472110.81, std=$25428.99
Southeast Second Half: n=6, Mean=$363775.88, std=$15386.82
Southeast Mean Difference: 108334.94
Southeast Standard Deviation Difference: 12133.89
Manual 95% CI: ($84552.52, $132117.36)
scipy 95% CI:  ($84552.95, $132116.93)
t-based 95% CI: ($81628.43, $135041.45)
t critical value at df=11: 2.2010
Effect Size: $108334.94
Effect Size Percentage: 22.95%
t-statistic: 8.9283
p-value: 0.0000044


## The Southeast question

Finding the Effect Size = $108334.94 at a 22.95% drop. 

At a 95% CI, the true drop could between $84,552.95 to $132,116.93.

With the p-value at 0.0000044, the chance of this drop appearing by chance is extremely unlikely. A zero difference is not included which agrees wit the p-value